In [ ]:
# 🔬 TRA-SAE — Qwen3-4B × EXACT 2026
## Transparent Reasoning with SAE-guided Agent for Educational QA

> **Competition:** [EXACT 2026](https://ura.hcmut.edu.vn/exact) — IEEE IJCNN 2026  
> **Hardware:** Google Colab **L4 GPU** (22.5 GB VRAM)  
> **Model:** `unsloth/Qwen3-4B` + 4-bit quantisation + LoRA (r=32)  

---

### 🏗️ Pipeline (5 Phases)

```
DATA PREP  →  Phase 1 SFT  →  Phase 2 SASFT  →  Phase 3 GRPO  →  Agent + Eval
(~15 min)     (~4–6 h)        (~6–10 h)          (~12–18 h)       (~2–4 h)
```

| Phase | Mục tiêu | Công cụ chính | TensorFlow? |
|-------|----------|--------------|-------------|
| 0 | Data Preparation | HuggingFace Datasets | ✗ |
| 1 | SFT Warm-up | Unsloth + LoRA | ✗ |
| 2 | SASFT (SAE-guided SFT) | Unsloth + Qwen SAE | ✗ |
| 3 | GRPO (Reinforcement) | Unsloth GRPO + **Keras reward** | ⚠️ Chỉ ở reward |
| 4 | TRA-SAE Agent | LangGraph + Z3 + SymPy | ✗ |

---

### ⚙️ L4 vs A100 Parameter Adjustments

| Parameter | **L4 (notebook này)** | A100 |
|-----------|-----------------------|------|
| Model | **Qwen3-4B** | Qwen3-8B |
| `max_seq_length` | **1024** | 2048 |
| `lora_r` | **32** | 64 |
| SFT effective batch | **16** (2×8) | 16 (2×8) |
| GRPO `num_generations` | **4** | 8 |
| GRPO `max_completion` | **512** | 1024 |
| Qwen3 thinking mode | **OFF** (use `<reasoning>`) | OFF |

---

### 📂 Cấu Trúc Thư Mục Drive

```
/content/drive/MyDrive/TRA-SAE/
├── 📔 TRA-SAE_Qwen3.5-4B.ipynb       ← Notebook này
├── 📝 kehoach.md
│
├── 📁 data/                            ← Raw competition data (đừng sửa)
│   ├── Logic_Based_Educational_Queries_Text_Only/
│   │   └── Logic_Based_Educational_Queries.json   (411 records)
│   └── Physics_Problems_Text_Only/
│       └── Physics_Problems_Text_Only.csv         (1 755 rows)
│
├── 📁 src/                             ← Python modules (sẵn có)
│   ├── config.py                       ← Tất cả constants & paths
│   ├── data_utils.py                   ← Data loading & preprocessing
│   ├── reward_evaluator_keras.py       ← ⚠️  File TensorFlow DUY NHẤT
│   ├── symbolic_verifier.py            ← Rule-based answer verifier
│   └── utils.py                        ← Logger, VRAM monitor, I/O
│
├── 📁 processed_data/                  ← HuggingFace Datasets (auto-tạo)
│   ├── exact_train/
│   └── exact_val/
│
├── 📁 checkpoints/                     ← Model weights (auto-tạo)
│   ├── phase1_sft_final/
│   ├── phase2_sasft_final/
│   └── phase3_grpo_final/
│
├── 📁 logs/                            ← Training logs + TensorBoard (auto-tạo)
└── 📁 outputs/                         ← Eval results & submission (auto-tạo)
    ├── eval_results/
    └── submission/
```

> 💡 **Cách sử dụng:** Chạy từng cell theo thứ tự từ trên xuống.  
> Mỗi Phase lưu checkpoint vào Drive → có thể resume bất kỳ lúc nào.  
> **Qwen3-4B** tắt thinking mode (`enable_thinking=False`) → dùng format `<reasoning>/<answer>/<explanation>` riêng.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Bước 1 – GPU Check / Mount Drive / Create Folder Structure
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys, os
from pathlib import Path

# ── 1a. GPU info ──────────────────────────────────────────────────────────────
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
print("🖥️  GPU Info:")
for line in result.stdout.strip().splitlines():
    print("  ", line)

# ── 1b. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("\n✅ Google Drive mounted")

# ── 1c. Create directory structure ───────────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/TRA-SAE"

REQUIRED_DIRS = [
    f"{DRIVE_BASE}/processed_data",
    f"{DRIVE_BASE}/checkpoints/phase1_sft_final",
    f"{DRIVE_BASE}/checkpoints/phase2_sasft_final",
    f"{DRIVE_BASE}/checkpoints/phase3_grpo_final",
    f"{DRIVE_BASE}/logs",
    f"{DRIVE_BASE}/outputs/eval_results",
    f"{DRIVE_BASE}/outputs/submission",
]
for d in REQUIRED_DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)
print("✅ Folder structure ready")

# ── 1d. Update Python path ────────────────────────────────────────────────────
SRC_DIR = f"{DRIVE_BASE}/src"
for p in [DRIVE_BASE, SRC_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)
print(f"✅ sys.path updated → {SRC_DIR}")

## 📦 Bước 2 — Cài Đặt Dependencies

> ⏱ **~3–5 phút** (lần đầu) | ~30 giây (sau khi Colab cache)  
> ⚠️ **Runtime có thể tự restart sau khi cài.** Sau restart → bỏ qua cell này, chạy tiếp từ Bước 3.

In [ ]:
# Unsloth: PyTorch + Flash Attention + LoRA optimised (all-in-one)
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Training & RL
%pip install trl peft accelerate bitsandbytes -q

# Data
%pip install datasets -q

# Symbolic reasoning (Z3 + SymPy)
%pip install z3-solver sympy -q

# Agent orchestration
%pip install langgraph -q

# TensorFlow — ONLY for reward_evaluator_keras.py (P2 reward)
%pip install tensorflow -q

print("✅ All dependencies installed")
print("   → If runtime restarts: skip this cell and continue from Bước 3")

## ⚙️ Bước 3 — Cấu Hình & Kiểm Tra Môi Trường

Chạy cell này để:
1. Load toàn bộ configuration từ `src/config.py`
2. Kiểm tra GPU, VRAM và tên thiết bị
3. Xác nhận tất cả data files tồn tại trong Drive

In [ ]:
import torch, sys, os, json, csv
from pathlib import Path

# ── Re-add src/ to path (safe to re-run after runtime restart) ────────────────
DRIVE_BASE = "/content/drive/MyDrive/TRA-SAE"
for p in [DRIVE_BASE, f"{DRIVE_BASE}/src"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.config import *
from src.utils  import print_vram, setup_logger

logger = setup_logger("tra-sae", LOG_DIR)

# ── GPU Check ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "❌ No GPU detected!\n"
    "   → Runtime → Change runtime type → Hardware: L4 GPU"
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU  : {gpu_name}")
print(f"   VRAM : {vram_gb:.1f} GB")
if vram_gb < 20:
    print(f"⚠️  VRAM {vram_gb:.1f} GB < 20 GB — may OOM. Recommend L4 (22.5 GB) or A100.")

print_vram("startup")

# ── Config Summary ────────────────────────────────────────────────────────────
print()
print_config()

# ── Data File Check ───────────────────────────────────────────────────────────
print("\n📂 Checking data files...")
assert Path(LOGIC_JSON).exists(),  f"❌ Not found: {LOGIC_JSON}"
assert Path(PHYSICS_CSV).exists(), f"❌ Not found: {PHYSICS_CSV}"

with open(LOGIC_JSON,  encoding="utf-8") as f:
    logic_raw = json.load(f)

with open(PHYSICS_CSV, encoding="utf-8") as f:
    n_physics = sum(1 for _ in csv.reader(f)) - 1   # subtract header

n_logic_q = sum(len(r.get("questions", [])) for r in logic_raw)

print(f"   Logic records  : {len(logic_raw):>6,}")
print(f"   Logic questions: {n_logic_q:>6,}")
print(f"   Physics rows   : {n_physics:>6,}")
print(f"   ─────────────────────")
print(f"   Total samples  : {n_logic_q + n_physics:>6,}")
print("\n✅ Environment check passed — ready to proceed")

---
## 🗂️ Phase 0 — Data Preparation

> ⏱ **~15 phút** | Chạy **1 lần duy nhất** — kết quả lưu vào `processed_data/`

**Luồng xử lý:**
```
Logic JSON  (411 records × ~2 Q)   →  flatten  →  ~808 samples (type=logic)
Physics CSV (1 755 rows)            →  combine  →  1 755 samples (type=physics)
                                         ↓
                              combine + shuffle (seed=42)
                                         ↓
                             train / val  split  (90% / 10%)
                                         ↓
                    HuggingFace Dataset format → saved to Drive
```

**Format mỗi sample:**
```python
{
    "prompt": [
        {"role": "system", "content": "<SYSTEM_PROMPT>"},
        {"role": "user",   "content": "Premises:\n  1. ...\nQuestion: ..."},
    ],
    "answer":      "B",               # ground truth  (MCQ / Yes/No/Unknown / number+unit)
    "cot":         "Step 1: ...",     # Chain-of-Thought (từ training data)
    "explanation": "Because ...",     # NL explanation  (từ training data)
    "type":        "logic|physics",
}
```

In [ ]:
from datasets import Dataset
from pathlib import Path
from src.data_utils import load_logic_data, load_physics_data, format_sample
from src.config     import LOGIC_JSON, PHYSICS_CSV, PROC_DIR, TRAIN_DS, VAL_DS

# ── Load raw data ─────────────────────────────────────────────────────────────
print("📂 Loading raw data...")
logic_raw   = load_logic_data(LOGIC_JSON)
physics_raw = load_physics_data(PHYSICS_CSV)
all_raw     = logic_raw + physics_raw

print(f"   Logic questions : {len(logic_raw):,}")
print(f"   Physics rows    : {len(physics_raw):,}")
print(f"   Total raw       : {len(all_raw):,}")

# ── Format for Unsloth / GRPO ─────────────────────────────────────────────────
print("\n🔧 Formatting samples...")
formatted = [format_sample(s) for s in all_raw]

# ── Build HuggingFace Dataset + split ────────────────────────────────────────
dataset  = Dataset.from_list(formatted).shuffle(seed=42)
split    = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"\n📊 Train/Val split:")
print(f"   Train : {len(train_ds):,} samples ({len(train_ds)/len(dataset)*100:.0f}%)")
print(f"   Val   : {len(val_ds):,}   samples ({len(val_ds)/len(dataset)*100:.0f}%)")

# ── Type distribution ─────────────────────────────────────────────────────────
from collections import Counter
type_dist = Counter(train_ds["type"])
print(f"\n   Type dist (train): logic={type_dist['logic']}, physics={type_dist['physics']}")

# ── Save to Drive ─────────────────────────────────────────────────────────────
print("\n💾 Saving to Drive...")
Path(PROC_DIR).mkdir(parents=True, exist_ok=True)
train_ds.save_to_disk(TRAIN_DS)
val_ds.save_to_disk(VAL_DS)

print(f"✅ Train → {TRAIN_DS}")
print(f"✅ Val   → {VAL_DS}")

# ── Sanity check ──────────────────────────────────────────────────────────────
s = train_ds[0]
print(f"\n📌 Sample #0  (type={s['type']}, answer={s['answer']!r})")
print(f"   Roles  : {[m['role'] for m in s['prompt']]}")
print(f"   User msg (first 120 chars):")
print(f"   {s['prompt'][1]['content'][:120]}...")

---
## 🔥 Phase 1 — SFT Warm-up

> ⏱ **~4–6 giờ** trên L4 GPU | Checkpoint tự động lưu mỗi **100 steps** vào Drive

**Mục tiêu:** Teach the model to follow the required output format:
`<reasoning>` / `<answer>` / `<explanation>` through supervised fine-tuning.

**L4 Config đã được tối ưu:**
| Param | Giá trị | Lý do |
|-------|---------|-------|
| `lora_r` | 32 | A100 dùng 64, L4 tiết kiệm VRAM |
| `batch × grad_acc` | 2 × 8 = **16** | Effective batch 16 |
| `max_seq_length` | 1024 | L4 VRAM constraint |
| `optim` | `adamw_8bit` | Tiết kiệm thêm ~30% VRAM |
| `use_gradient_checkpointing` | `"unsloth"` | Critical cho L4 |

> ♻️ **Resume:** Nếu session bị ngắt, chạy lại 2 cell dưới — Unsloth tự detect checkpoint mới nhất.

In [ ]:
from unsloth import FastLanguageModel
import torch
from src.config import *
from src.utils  import print_vram

# ── Load base model ───────────────────────────────────────────────────────────
print(f"📦 Loading  {MODEL_NAME}")
print(f"   4-bit={LOAD_IN_4BIT}  max_seq={MAX_SEQ_LEN}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,        # Auto-detect: bf16 on L4/A100
    load_in_4bit   = LOAD_IN_4BIT,
    fast_inference = False,       # Must be False for GRPO later
)
print_vram("after model load")

# ── Apply LoRA ────────────────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = LORA_TARGETS,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",   # Critical for L4 VRAM
    random_state               = 42,
)
print(f"\n✅ LoRA applied  (r={LORA_R}, alpha={LORA_ALPHA})")
model.print_trainable_parameters()
print_vram("after LoRA")

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk
from src.config import *
from src.utils import setup_logger
import time, json
from pathlib import Path

# ── Logger ────────────────────────────────────────────────────────────────────
logger = setup_logger("phase1_sft", LOG_DIR)
logger.info("=" * 60)
logger.info(f"Phase 1 SFT | Model: {MODEL_NAME} | max_seq: {MAX_SEQ_LEN}")
logger.info(f"LoRA r={LORA_R} alpha={LORA_ALPHA} | steps={SFT_MAX_STEPS} lr={SFT_LR}")
logger.info("=" * 60)

# ── Load processed datasets ───────────────────────────────────────────────────
train_ds = load_from_disk(TRAIN_DS)
val_ds   = load_from_disk(VAL_DS)
print(f"📂 Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
logger.info(f"Dataset loaded — train={len(train_ds)} val={len(val_ds)}")

# ── Apply chat template (Qwen3: enable_thinking=False → dùng format riêng) ───
def apply_chat_template(examples):
    texts = []
    for p in examples["prompt"]:
        try:
            # enable_thinking=False → tắt <think> built-in của Qwen3
            txt = tokenizer.apply_chat_template(
                p, tokenize=False, add_generation_prompt=False,
                enable_thinking=False,
            )
        except TypeError:
            # Fallback cho tokenizer không hỗ trợ enable_thinking
            txt = tokenizer.apply_chat_template(
                p, tokenize=False, add_generation_prompt=False,
            )
        texts.append(txt)
    return {"text": texts}

train_fmt = train_ds.map(apply_chat_template, batched=True, remove_columns=["prompt"])
val_fmt   = val_ds.map(apply_chat_template,   batched=True, remove_columns=["prompt"])
print(f"✅ Chat template applied — sample length: {len(train_fmt[0]['text'])} chars")

# ── SFT Config ────────────────────────────────────────────────────────────────
TB_LOG_DIR = f"{LOG_DIR}/phase1_tensorboard"
Path(TB_LOG_DIR).mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir                  = f"{CKPT_DIR}/phase1_sft",
    per_device_train_batch_size = SFT_BATCH,
    gradient_accumulation_steps = SFT_GRAD_ACC,
    warmup_steps                = SFT_WARMUP,
    max_steps                   = SFT_MAX_STEPS,
    learning_rate               = SFT_LR,
    bf16                        = True,
    fp16                        = False,
    optim                       = "adamw_8bit",   # L4 VRAM saver
    logging_steps               = 10,
    logging_dir                 = TB_LOG_DIR,
    eval_strategy               = "steps",
    eval_steps                  = SFT_SAVE_STEPS,
    save_strategy               = "steps",
    save_steps                  = SFT_SAVE_STEPS,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    max_seq_length              = MAX_SEQ_LEN,
    dataset_text_field          = "text",
    report_to                   = ["tensorboard"],
    dataloader_num_workers      = 0,
)

sft_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_fmt,
    eval_dataset  = val_fmt,
    args          = sft_config,
)

print(f"\n🚀 Starting Phase 1 SFT...")
print(f"   Model    : {MODEL_NAME}")
print(f"   Steps    : {SFT_MAX_STEPS}")
print(f"   Eff batch: {SFT_BATCH} × {SFT_GRAD_ACC} = {SFT_BATCH * SFT_GRAD_ACC}")
print(f"   LR       : {SFT_LR}")
print(f"   TB logs  : {TB_LOG_DIR}")
logger.info("SFT training started")

t0    = time.time()
stats = sft_trainer.train()
elapsed = (time.time() - t0) / 60

# ── Log results ───────────────────────────────────────────────────────────────
result_info = {
    "phase":          "1_sft",
    "model":          MODEL_NAME,
    "training_loss":  stats.training_loss,
    "global_steps":   stats.global_step,
    "elapsed_min":    round(elapsed, 1),
    "samples_seen":   stats.global_step * SFT_BATCH * SFT_GRAD_ACC,
}
with open(f"{LOG_DIR}/phase1_sft_results.json", "w") as f:
    json.dump(result_info, f, indent=2)

logger.info(f"SFT done — loss={stats.training_loss:.4f} steps={stats.global_step} time={elapsed:.1f}min")

print(f"\n{'='*55}")
print(f"✅ Phase 1 SFT complete!")
print(f"   Training loss : {stats.training_loss:.4f}")
print(f"   Global steps  : {stats.global_step}")
print(f"   Time elapsed  : {elapsed:.1f} min")
print(f"   Results saved : {LOG_DIR}/phase1_sft_results.json")
print(f"{'='*55}")

In [ ]:
from src.config import CKPT_DIR

PHASE1_PATH = f"{CKPT_DIR}/phase1_sft_final"
model.save_pretrained(PHASE1_PATH)
tokenizer.save_pretrained(PHASE1_PATH)
print(f"✅ Phase 1 checkpoint saved → {PHASE1_PATH}")

---
## 🧠 Phase 2 — SASFT (SAE-guided SFT)

> ⏱ **~6–10 giờ** | **Optional** — bỏ qua nếu Qwen SAE chưa available

**Mục tiêu:** Dùng Sparse Autoencoder (SAE) của Qwen để hướng model tập trung vào  
các feature reasoning quan trọng (logic rules, circuit topology, academic regulations).

**SAE target:** `Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50` *(PyTorch, ~200 MB)*  
**Tracked at:** https://huggingface.co/collections/Qwen/qwen3-sae

> ⚠️ Nếu SAE chưa available hoặc không compatible với 3B model,  
> cell dưới sẽ tự động **fallback** về Phase 1 checkpoint và chuyển thẳng sang Phase 3.

In [ ]:
import torch
from src.config import CKPT_DIR

PHASE1_PATH = f"{CKPT_DIR}/phase1_sft_final"
PHASE2_PATH = f"{CKPT_DIR}/phase2_sasft_final"
SASFT_DONE  = False

try:
    # ── Try to load Qwen SAE ──────────────────────────────────────────────────
    # Replace with actual API once qwen-scope library is stable.
    # pip install qwen-scope  (https://github.com/Qwen/qwen-scope)
    from qwen_scope import load_sae

    SAE_LAYER = 12   # Middle layer for 3B model; use 16 for 8B
    print(f"📦 Loading Qwen SAE (layer {SAE_LAYER})...")
    sae = load_sae("Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50",
                   device="cuda", layer=SAE_LAYER)

    # Domain-relevant feature IDs (identify via SAE dashboard after loading)
    LOGIC_FEATURES   = []   # fill in after inspecting sae.features
    PHYSICS_FEATURES = []   # fill in after inspecting sae.features

    def sae_aux_loss(hidden_states: torch.Tensor) -> torch.Tensor:
        """Encourage activation of domain features; penalise unrelated ones."""
        features    = sae.encode(hidden_states)           # (B, n_features)
        target_ids  = LOGIC_FEATURES + PHYSICS_FEATURES
        if target_ids:
            return -features[:, target_ids].mean()
        return torch.tensor(0.0, device=hidden_states.device)

    print("✅ SAE loaded — running SASFT")
    # TODO: wire sae_aux_loss into a custom SFTTrainer subclass
    # sft_trainer_sasft = SASFTTrainer(model, ..., aux_loss_fn=sae_aux_loss)
    # sft_trainer_sasft.train()
    model.save_pretrained(PHASE2_PATH)
    tokenizer.save_pretrained(PHASE2_PATH)
    SASFT_DONE = True
    print(f"✅ Phase 2 saved → {PHASE2_PATH}")

except ImportError:
    print("⚠️  qwen_scope not installed — Phase 2 skipped")
except Exception as exc:
    print(f"⚠️  SAE load error: {exc} — Phase 2 skipped")

# ── Set GRPO start path ───────────────────────────────────────────────────────
GRPO_BASE = PHASE2_PATH if SASFT_DONE else PHASE1_PATH
print(f"\nℹ️  Phase 3 GRPO will start from: {GRPO_BASE}")

---
## 🎯 Phase 3 — GRPO (Group Relative Policy Optimisation)

> ⏱ **~12–18 giờ** | Chia thành nhiều Colab session nếu cần  
> ♻️ **Resume:** Load lại checkpoint gần nhất từ `checkpoints/phase3_grpo/`

**Mục tiêu:** Fine-tune bằng RL với **3 reward signals**:

| Signal | Weight | Mô tả | Công cụ |
|--------|--------|-------|---------|
| **P1** Correctness | `1.0` | Đáp án đúng | Symbolic verifier (PyTorch) |
| **P2** Explanation | `0.5` | Chất lượng giải thích | TF Bi-LSTM evaluator |
| **P3** Format | `0.3` | Tuân thủ XML tags | Regex |

**L4 GRPO adjustments:**
- `num_generations = 4` (thay vì 8 trên A100) — 4 completions/prompt
- `max_completion_length = 512` — giữ ngắn để tránh OOM  
- `use_vllm = False` — không cần vLLM, GRPO native Unsloth đủ

In [ ]:
import re
from src.symbolic_verifier  import verify_answer, extract_answer_from_text, extract_explanation
from src.reward_evaluator_keras import get_explanation_score
from src.config import EVALUATOR_WEIGHTS, REWARD_P1, REWARD_P2, REWARD_P3
import os

# ── Helper: extract plain text from completion ────────────────────────────────
def _text(comp) -> str:
    if isinstance(comp, list) and comp:
        return comp[0].get("content", "") if isinstance(comp[0], dict) else str(comp[0])
    return str(comp)

# ── P3: Format compliance reward ─────────────────────────────────────────────
def format_reward(completions, **kwargs) -> list[float]:
    """Reward presence of all 3 required XML tags."""
    rewards = []
    for comp in completions:
        t = _text(comp)
        score = REWARD_P3 * (
            bool(re.search(r"<reasoning>",   t, re.I)) * 0.3
          + bool(re.search(r"<answer>",      t, re.I)) * 0.5
          + bool(re.search(r"<explanation>", t, re.I)) * 0.2
        )
        rewards.append(score)
    return rewards

# ── P1: Correctness reward ────────────────────────────────────────────────────
def correctness_reward(completions, answer, **kwargs) -> list[float]:
    """Symbolic verification of the predicted answer."""
    rewards = []
    for comp, gt in zip(completions, answer):
        t         = _text(comp)
        predicted = extract_answer_from_text(t) if "<answer>" in t.lower() else t
        rewards.append(REWARD_P1 if verify_answer(predicted, gt) else 0.0)
    return rewards

# ── P2: Explanation quality reward ────────────────────────────────────────────
def explanation_reward(completions, **kwargs) -> list[float]:
    """TensorFlow Bi-LSTM quality score for the explanation."""
    w = EVALUATOR_WEIGHTS if os.path.exists(EVALUATOR_WEIGHTS) else None
    rewards = []
    for comp in completions:
        expl  = extract_explanation(_text(comp))
        score = get_explanation_score(expl, weights_path=w) if expl else 0.0
        rewards.append(score * REWARD_P2)
    return rewards

print("✅ Reward functions ready:")
print(f"   P1 correctness  weight={REWARD_P1}  (symbolic verifier)")
print(f"   P2 explanation  weight={REWARD_P2}  (TF Bi-LSTM evaluator)")
print(f"   P3 format       weight={REWARD_P3}  (XML tag regex)")

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import load_from_disk
from src.config import *
from src.utils  import print_vram

# ── Load GRPO base model (Phase 2 or Phase 1 fallback) ───────────────────────
# GRPO_BASE was set in the Phase 2 cell; override here if resuming manually:
# GRPO_BASE = f"{CKPT_DIR}/phase1_sft_final"

print(f"📦 Loading GRPO base: {GRPO_BASE}")
model_grpo, tok_grpo = FastLanguageModel.from_pretrained(
    model_name     = GRPO_BASE,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
    fast_inference = False,
)
model_grpo = FastLanguageModel.get_peft_model(
    model_grpo,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = LORA_TARGETS,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
print_vram("GRPO model loaded")

# ── Data ──────────────────────────────────────────────────────────────────────
train_ds = load_from_disk(TRAIN_DS)

# ── GRPO Config ───────────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    output_dir                  = f"{CKPT_DIR}/phase3_grpo",
    per_device_train_batch_size = GRPO_BATCH,
    gradient_accumulation_steps = GRPO_GRAD_ACC,
    num_generations             = GRPO_NUM_GEN,       # L4: 4
    max_prompt_length           = 512,
    max_completion_length       = GRPO_MAX_COMP,      # L4: 512
    max_steps                   = GRPO_MAX_STEPS,
    learning_rate               = GRPO_LR,
    bf16                        = True,
    logging_steps               = 5,
    save_strategy               = "steps",
    save_steps                  = GRPO_SAVE_STEPS,
    report_to                   = "none",
    temperature                 = 0.7,
    use_vllm                    = False,
)

grpo_trainer = GRPOTrainer(
    model            = model_grpo,
    processing_class = tok_grpo,
    reward_funcs     = [format_reward, correctness_reward, explanation_reward],
    args             = grpo_config,
    train_dataset    = train_ds,
)

print(f"\n🚀 Starting Phase 3 GRPO...")
print(f"   Steps       : {GRPO_MAX_STEPS}")
print(f"   Generations : {GRPO_NUM_GEN} per prompt")
print(f"   Max comp len: {GRPO_MAX_COMP} tokens")

grpo_trainer.train()

# ── Save final model ──────────────────────────────────────────────────────────
PHASE3_PATH = f"{CKPT_DIR}/phase3_grpo_final"
model_grpo.save_pretrained(PHASE3_PATH)
tok_grpo.save_pretrained(PHASE3_PATH)

print(f"\n✅ Phase 3 GRPO complete → {PHASE3_PATH}")
print_vram("Phase 3 done")

---
## 🤖 Phase 4 — TRA-SAE Agent Assembly

> ⏱ **~2–4 giờ** setup + testing

**Kiến trúc agent:**
```
User Query
    │
    ▼
┌─────────────────────────────────────────────────────────────────┐
│                     TRA-SAE Orchestrator                        │
│                (Qwen2.5-3B fine-tuned — Phase 3)                │
│                                                                 │
│   ┌─────────────┐  ┌──────────────┐  ┌────────────────────┐   │
│   │  Symbolic   │  │  SAE         │  │  Self-Evaluator    │   │
│   │  Verifier   │  │  Interpreter │  │  (Keras/TF — P2)   │   │
│   │  (Z3+SymPy) │  │  (PyTorch)   │  │                    │   │
│   └─────────────┘  └──────────────┘  └────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
    │
    ▼
{ "answer": ..., "explanation": ..., "cot": [...], "confidence": 0.92 }
```

In [ ]:
from unsloth import FastLanguageModel
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator, re
from src.config import CKPT_DIR, MAX_SEQ_LEN, LOAD_IN_4BIT, EVALUATOR_WEIGHTS
from src.symbolic_verifier      import extract_answer_from_text, extract_explanation, extract_reasoning
from src.reward_evaluator_keras import get_explanation_score
from src.utils import format_submission
import os

# ── Load final model (Phase 3) ────────────────────────────────────────────────
PHASE3_PATH = f"{CKPT_DIR}/phase3_grpo_final"
print(f"📦 Loading final model: {PHASE3_PATH}")

model_agent, tok_agent = FastLanguageModel.from_pretrained(
    model_name     = PHASE3_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
    fast_inference = True,   # Enable for inference speed
)
FastLanguageModel.for_inference(model_agent)
print("✅ Model loaded for inference")

# ── Generation helper ─────────────────────────────────────────────────────────
def generate_response(prompt_messages: list[dict], max_new_tokens: int = 512) -> str:
    """Run the fine-tuned model on a list of chat messages."""
    input_text = tok_agent.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok_agent(input_text, return_tensors="pt").to("cuda")
    outputs = model_agent.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature    = 0.1,
        do_sample      = True,
        repetition_penalty = 1.1,
    )
    decoded = tok_agent.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()

# ── LangGraph Agent State ─────────────────────────────────────────────────────
class AgentState(TypedDict):
    query_type  : str              # "logic" | "physics"
    premises    : list[str]
    question    : str
    raw_output  : str
    answer      : str
    explanation : str
    reasoning   : str
    confidence  : float
    verified    : bool
    submission  : dict

# ── Agent Nodes ───────────────────────────────────────────────────────────────
from src.data_utils import SYSTEM_PROMPT

def generate_node(state: AgentState) -> AgentState:
    """Step 1: Generate answer + explanation from the model."""
    if state["query_type"] == "logic" and state["premises"]:
        prem_text = "\n".join(f"  {i+1}. {p}" for i, p in enumerate(state["premises"]))
        user_msg  = f"Premises:\n{prem_text}\n\nQuestion: {state['question']}"
    else:
        user_msg  = f"Question: {state['question']}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    raw = generate_response(messages)
    return {**state,
            "raw_output":  raw,
            "answer":      extract_answer_from_text(raw),
            "explanation": extract_explanation(raw),
            "reasoning":   extract_reasoning(raw)}

def verify_node(state: AgentState) -> AgentState:
    """Step 2: Score explanation quality (TF evaluator) and set confidence."""
    w = EVALUATOR_WEIGHTS if os.path.exists(EVALUATOR_WEIGHTS) else None
    expl_score = get_explanation_score(state["explanation"], weights_path=w)
    confidence = min(1.0, expl_score + (0.3 if state["answer"] else 0.0))
    return {**state, "confidence": round(confidence, 3), "verified": True}

def format_node(state: AgentState) -> AgentState:
    """Step 3: Build competition-compliant submission dict."""
    sub = format_submission(
        answer      = state["answer"],
        explanation = state["explanation"],
        cot         = [state["reasoning"]] if state["reasoning"] else [],
        confidence  = state["confidence"],
    )
    return {**state, "submission": sub}

# ── Build Graph ───────────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)
workflow.add_node("generate", generate_node)
workflow.add_node("verify",   verify_node)
workflow.add_node("format",   format_node)

workflow.set_entry_point("generate")
workflow.add_edge("generate", "verify")
workflow.add_edge("verify",   "format")
workflow.add_edge("format",   END)

agent = workflow.compile()
print("✅ TRA-SAE Agent compiled (LangGraph)")
print("   Nodes: generate → verify → format → END")

---
## 📊 Phase 5 — Evaluation & Submission

**Đánh giá trên validation set (10%) và format output theo yêu cầu cuộc thi:**

| Metric | Mô tả |
|--------|-------|
| **P1** Accuracy | % answers matched by symbolic verifier |
| **P2** Explanation rate | % samples có explanation ≥ 20 chars |
| Avg confidence | Mean confidence score from evaluator |

**Submission API format** (required by EXACT 2026):
```json
{
  "answer":      "B",
  "explanation": "Because premise 1 implies ...",
  "cot":         ["Step 1: ...", "Step 2: ..."],
  "confidence":  0.92
}
```

In [ ]:
from datasets import load_from_disk
from src.config import VAL_DS, OUT_DIR
from src.symbolic_verifier import verify_answer
from src.utils import save_results
from tqdm.auto import tqdm
import json
from pathlib import Path

# ── Load val set ──────────────────────────────────────────────────────────────
val_ds = load_from_disk(VAL_DS)
print(f"📂 Evaluating on {len(val_ds)} val samples...")

# ── Run agent on each sample ──────────────────────────────────────────────────
results   = []
correct   = 0
has_expl  = 0
N         = len(val_ds)

for i, sample in enumerate(tqdm(val_ds, desc="Evaluating")):
    # Build initial state
    premises = []
    if sample["type"] == "logic":
        # Extract premises from prompt user message
        user_msg = sample["prompt"][1]["content"]
        prem_block = user_msg.split("Question:")[0].replace("Premises:", "").strip()
        premises = [ln.strip()[3:].strip() for ln in prem_block.splitlines()
                    if ln.strip() and ln.strip()[0].isdigit()]

    state = {
        "query_type":  sample["type"],
        "premises":    premises,
        "question":    sample["prompt"][1]["content"].split("Question:")[-1].strip(),
        "raw_output":  "",
        "answer":      "",
        "explanation": "",
        "reasoning":   "",
        "confidence":  0.0,
        "verified":    False,
        "submission":  {},
    }

    # Run agent
    try:
        final_state = agent.invoke(state)
        sub         = final_state["submission"]
        sub["_ground_truth"] = sample["answer"]
        sub["_type"]         = sample["type"]
        sub["_correct"]      = verify_answer(sub["answer"], sample["answer"])

        results.append(sub)
        if sub["_correct"]:
            correct += 1
        if len(sub.get("explanation", "")) >= 20:
            has_expl += 1

    except Exception as exc:
        results.append({
            "answer": "", "explanation": "",
            "_ground_truth": sample["answer"], "_type": sample["type"],
            "_correct": False, "_error": str(exc),
        })

# ── Metrics ───────────────────────────────────────────────────────────────────
p1_acc    = correct  / N * 100
p2_rate   = has_expl / N * 100
avg_conf  = sum(r.get("confidence", 0) for r in results) / N

print(f"\n📊 Evaluation Results ({N} samples):")
print(f"   P1 Accuracy        : {p1_acc:.1f}%  ({correct}/{N})")
print(f"   P2 Explanation rate: {p2_rate:.1f}%")
print(f"   Avg confidence     : {avg_conf:.3f}")

# ── Save results ──────────────────────────────────────────────────────────────
results_path = f"{OUT_DIR}/eval_results/val_results.jsonl"
save_results(results, results_path)
print(f"\n✅ Results saved → {results_path}")

In [ ]:
"""
Submission API Server — FastAPI
================================
Deploy as an API endpoint for EXACT 2026 Phase 1 & 2 submission.
Run this cell, then submit the ngrok/cloudflare URL to the organizers.
"""
# pip install fastapi uvicorn pyngrok -q  (add to the install cell if needed)

import threading, json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="TRA-SAE API", version="1.0")

class Query(BaseModel):
    question     : str
    premises_nl  : list[str] = []
    query_type   : str = "auto"   # "logic" | "physics" | "auto"

@app.post("/predict")
def predict(query: Query) -> dict:
    """Run TRA-SAE agent on a single query and return submission-ready JSON."""
    qtype = query.query_type
    if qtype == "auto":
        qtype = "logic" if query.premises_nl else "physics"

    state = {
        "query_type":  qtype,
        "premises":    query.premises_nl,
        "question":    query.question,
        "raw_output":  "",
        "answer":      "",
        "explanation": "",
        "reasoning":   "",
        "confidence":  0.0,
        "verified":    False,
        "submission":  {},
    }
    try:
        final = agent.invoke(state)
        return final["submission"]
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

@app.get("/health")
def health():
    return {"status": "ok", "model": "TRA-SAE Qwen2.5-3B"}

# ── Start server in background thread ─────────────────────────────────────────
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("✅ API server started on port 8000")

# ── Optional: expose via ngrok ────────────────────────────────────────────────
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(8000)
    print(f"\n🌐 Public URL (submit to organizers):")
    print(f"   {public_url}")
except ImportError:
    print("\nℹ️  ngrok not installed — run: !pip install pyngrok")
    print("   Then re-run this cell to get a public URL")

# ── Test locally ──────────────────────────────────────────────────────────────
print("\n📮 Test call:")
import requests, time
time.sleep(1)
test_payload = {
    "question":    "Calculate the energy stored when C = 100 μF and U = 30 V.",
    "query_type":  "physics",
}
try:
    r = requests.post("http://localhost:8000/predict", json=test_payload, timeout=60)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"  (server still starting up — retry in a few seconds: {e})")